# Figure 4 Weighted Descriptor Panels

This notebook regenerates the three weighted-descriptor projection panels from the curated 112-sample BO descriptor table.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

base = Path.cwd()
data_path = base / "curated" / "figure4_bo_weighted_ml_top3_descriptors.csv"
output_dir = base / "outputs"
output_dir.mkdir(exist_ok=True)

df = pd.read_csv(data_path)

required = [
    "weighted_electronegativity",
    "weighted_FE_ads_avg",
    "weighted_delta_EOH",
    "separation_coefficient",
]
missing = [col for col in required if col not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

plot_df = df.dropna(subset=required).copy()

mpl.rcParams["font.family"] = "Arial"
mpl.rcParams["font.weight"] = "bold"
mpl.rcParams["axes.linewidth"] = 2.5
mpl.rcParams["xtick.labelsize"] = 10
mpl.rcParams["ytick.labelsize"] = 10
mpl.rcParams["axes.labelsize"] = 12
mpl.rcParams["axes.labelweight"] = "bold"
mpl.rcParams["xtick.major.width"] = 1.0
mpl.rcParams["ytick.major.width"] = 1.0
mpl.rcParams["xtick.major.size"] = 5
mpl.rcParams["ytick.major.size"] = 5

BIN_EDGES = np.array([2, 4, 6, 8, 10, 12], dtype=float)
BIN_LABELS = [f"{int(BIN_EDGES[i])}-{int(BIN_EDGES[i + 1])}" for i in range(len(BIN_EDGES) - 1)]
BIN_COLORS = ["#AE5CFA", "#AAC0FC", "#F1E2AD", "#F19FA0", "#F35656"]
EDGE_COLOR = "black"
SCATTER_SIZE = 50
SCATTER_LW = 0.8
SCATTER_ALPHA = 0.9
HIST_ALPHA = 0.85
BAR_LW = 0.7
RIGHT_BAR_SCALE = 0.30

LABELS = {
    "weighted_electronegativity": "Weighted Electronegativity",
    "weighted_FE_ads_avg": "Weighted Adsorption Energy of H (eV)",
    "weighted_delta_EOH": r"Weighted $\Delta E_{\mathrm{OH}}$ (eV)",
}

OUTPUTS = [
    (
        "weighted_electronegativity",
        "weighted_FE_ads_avg",
        "joint_weighted_EN_vs_FEadsH_sepcoef_stacked_marginals.png",
    ),
    (
        "weighted_electronegativity",
        "weighted_delta_EOH",
        "joint_weighted_EN_vs_deltaEOH_sepcoef_stacked_marginals.png",
    ),
    (
        "weighted_delta_EOH",
        "weighted_FE_ads_avg",
        "joint_weighted_deltaEOH_vs_FEadsH_sepcoef_stacked_marginals.png",
    ),
]


def separation_bin_index(values):
    idx = np.digitize(values, BIN_EDGES, right=False) - 1
    return np.clip(idx, 0, len(BIN_EDGES) - 2)


def stacked_hist_by_color(ax, groups, bins, orientation="vertical", scale=1.0):
    weights = [np.ones_like(group, dtype=float) * scale for group in groups]
    ax.hist(
        groups,
        bins=bins,
        orientation=orientation,
        stacked=True,
        color=BIN_COLORS,
        edgecolor=EDGE_COLOR,
        linewidth=BAR_LW,
        alpha=HIST_ALPHA,
        weights=weights,
    )


def draw_joint_panel(x_col, y_col, out_name):
    x = plot_df[x_col].to_numpy(dtype=float)
    y = plot_df[y_col].to_numpy(dtype=float)
    sep = plot_df["separation_coefficient"].to_numpy(dtype=float)
    mask = np.isfinite(x) & np.isfinite(y) & np.isfinite(sep)
    x, y, sep = x[mask], y[mask], sep[mask]
    bin_idx = separation_bin_index(sep)

    fig = plt.figure(figsize=(5.0, 4.0), dpi=300)
    gs = fig.add_gridspec(
        2,
        2,
        height_ratios=[0.70, 4.00],
        width_ratios=[4.00, 0.80],
        hspace=0.03,
        wspace=0.03,
    )
    ax_histx = fig.add_subplot(gs[0, 0])
    ax_scatt = fig.add_subplot(gs[1, 0], sharex=ax_histx)
    ax_histy = fig.add_subplot(gs[1, 1], sharey=ax_scatt)
    ax_empty = fig.add_subplot(gs[0, 1])
    ax_empty.axis("off")

    for i, color in enumerate(BIN_COLORS):
        keep = bin_idx == i
        ax_scatt.scatter(
            x[keep],
            y[keep],
            s=SCATTER_SIZE,
            facecolors=color,
            edgecolors=EDGE_COLOR,
            linewidths=SCATTER_LW,
            alpha=SCATTER_ALPHA,
            zorder=3,
        )

    ax_scatt.set_xlabel(LABELS[x_col])
    ax_scatt.set_ylabel(LABELS[y_col])
    ax_scatt.grid(False)

    bins_x = np.linspace(np.nanmin(x), np.nanmax(x), 28)
    bins_y = np.linspace(np.nanmin(y), np.nanmax(y), 24)
    x_groups = [x[bin_idx == i] for i in range(len(BIN_COLORS))]
    y_groups = [y[bin_idx == i] for i in range(len(BIN_COLORS))]

    stacked_hist_by_color(ax_histx, x_groups, bins_x, orientation="vertical", scale=1.0)
    raw_y_max = max(
        np.histogram(group, bins=bins_y)[0].max() if len(group) else 0
        for group in y_groups
    )
    raw_y_max = max(raw_y_max, 1)
    stacked_hist_by_color(ax_histy, y_groups, bins_y, orientation="horizontal", scale=RIGHT_BAR_SCALE)
    ax_histy.set_xlim(0, raw_y_max * 1.05)

    ax_histx.tick_params(axis="x", labelbottom=False, bottom=False)
    ax_histx.tick_params(axis="y", left=False, labelleft=False)
    ax_histy.tick_params(axis="x", bottom=False, labelbottom=False)
    ax_histy.tick_params(axis="y", left=False, labelleft=False)

    for side in ["left", "right", "top"]:
        ax_histx.spines[side].set_visible(False)
    ax_histx.spines["bottom"].set_visible(True)
    ax_histx.spines["bottom"].set_linewidth(0.8)

    for side in ["right", "top", "bottom"]:
        ax_histy.spines[side].set_visible(False)
    ax_histy.spines["left"].set_visible(True)
    ax_histy.spines["left"].set_linewidth(0.8)

    for side in ["left", "right", "top", "bottom"]:
        ax_scatt.spines[side].set_linewidth(2.0)

    fig.savefig(output_dir / out_name, dpi=600, bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    print(f"Saved: {output_dir / out_name}")


for x_col, y_col, out_name in OUTPUTS:
    draw_joint_panel(x_col, y_col, out_name)

print(f"Plotted {len(plot_df)} BO samples from {data_path}")


Saved: C:\Users\yibinjiang\Documents\GitHub\HD_separation\0-Figures\Figure4\outputs\joint_weighted_EN_vs_FEadsH_sepcoef_stacked_marginals.png


Saved: C:\Users\yibinjiang\Documents\GitHub\HD_separation\0-Figures\Figure4\outputs\joint_weighted_EN_vs_deltaEOH_sepcoef_stacked_marginals.png


Saved: C:\Users\yibinjiang\Documents\GitHub\HD_separation\0-Figures\Figure4\outputs\joint_weighted_deltaEOH_vs_FEadsH_sepcoef_stacked_marginals.png
Plotted 112 BO samples from C:\Users\yibinjiang\Documents\GitHub\HD_separation\0-Figures\Figure4\curated\figure4_bo_weighted_ml_top3_descriptors.csv
